# 🎵 MoodBeats AI — Session 5: Detect Page & POST-Redirect-GET

Today we build the most important feature: **submitting a mood and getting Gemini's analysis**.

---
## The POST-Redirect-GET Pattern

```
1. Browser fills form and submits → POST /detect
2. FastAPI receives form data, calls Gemini
3. Stores result in memory cache with a UUID key
4. Returns HTTP 303 → redirect to GET /results?key=<uuid>
5. GET /results reads the UUID, fetches result, renders page
```

**Why redirect?** If the user refreshes /results, they get the cached result — not a duplicate API call.

---

In [ ]:
!pip install fastapi uvicorn jinja2 python-multipart nest-asyncio google-generativeai -q
import nest_asyncio; nest_asyncio.apply()
print("Ready!")

## Part 1: The In-Memory Result Cache

We can't store 8 songs in a cookie (4KB limit). Instead:  
- After Gemini responds, store the result in a Python dict with a UUID key
- Redirect the browser to `/results?key=<uuid>`
- The results page reads the UUID and retrieves the result

In [ ]:
import uuid
import time

_cache = {}   # { uuid_string: (result_dict, timestamp) }
_TTL = 3600   # 1 hour in seconds

def cache_store(result: dict) -> str:
    """Store result and return a UUID key."""
    key = str(uuid.uuid4())
    _cache[key] = (result, time.monotonic())
    return key

def cache_retrieve(key: str) -> dict | None:
    """Retrieve result by key, or None if expired/missing."""
    entry = _cache.get(key)
    if not entry:
        return None
    result, ts = entry
    if time.monotonic() - ts > _TTL:
        del _cache[key]
        return None
    return result

# Test the cache
test_result = {"emotion": "happy", "confidence": 0.9, "songs": []}
key = cache_store(test_result)
print(f"Stored with key: {key}")

retrieved = cache_retrieve(key)
print(f"Retrieved: {retrieved}")

missing = cache_retrieve("fake-uuid")
print(f"Missing key returns: {missing}")

### ✏️ In-Class Exercise 5a — Why UUID?

UUID stands for Universally Unique Identifier. It's a random 128-bit number.

In [ ]:
# Generate 5 UUIDs and observe they're all different
for _ in range(5):
    print(uuid.uuid4())

# Q: Why not use sequential IDs like 1, 2, 3?
# A: ___  (write your answer in a markdown cell)

---
## Part 2: HTML Forms and POST Requests

```html
<form method="POST" action="/detect">
    <textarea name="mood_text"></textarea>
    <button type="submit">Detect</button>
</form>
```

FastAPI reads form fields with `Form(...)`:

In [ ]:
import os, json, re
os.makedirs("templates", exist_ok=True)

# Detect template with a working form
with open("templates/detect.html", "w") as f:
    f.write("""
<!DOCTYPE html>
<html><head>
<title>Detect Mood</title>
<script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-gray-950 text-white min-h-screen p-10">
<h1 class="text-3xl font-bold text-indigo-400 mb-6">Detect Your Mood</h1>

{% if error %}
<div class="bg-red-900 text-red-300 p-4 rounded-lg mb-6">⚠️ {{ error }}</div>
{% endif %}

<form method="POST" action="/detect">
    <textarea name="mood_text"
        class="w-full bg-gray-800 border border-gray-700 rounded-lg p-4 text-white h-32 mb-4"
        placeholder="Describe how you're feeling..."></textarea>
    <button type="submit"
        class="bg-indigo-600 hover:bg-indigo-500 text-white px-6 py-3 rounded-lg font-semibold">
        Detect My Mood
    </button>
</form>
</body></html>
""")

# Results template
with open("templates/results.html", "w") as f:
    f.write("""
<!DOCTYPE html>
<html><head>
<title>Results</title>
<script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-gray-950 text-white min-h-screen p-10">
<h1 class="text-3xl font-bold mb-2">Your Mood: {{ result.emotion | title }}</h1>
<p class="text-gray-400 mb-2">Confidence: {{ (result.confidence * 100) | round | int }}%</p>
<p class="text-gray-300 mb-8">{{ result.mood_description }}</p>

<h2 class="text-xl font-semibold mb-4">Recommended Songs</h2>
<div class="grid grid-cols-1 md:grid-cols-2 gap-4">
{% for song in result.songs %}
<div class="bg-gray-800 rounded-lg p-4">
    <p class="font-semibold">{{ song.title }}</p>
    <p class="text-gray-400 text-sm">{{ song.artist }} · {{ song.genre }}</p>
</div>
{% endfor %}
</div>

<a href="/detect" class="mt-8 inline-block text-indigo-400 hover:underline">← Try Again</a>
</body></html>
""")

print("Templates written!")

---
## Part 3: The Complete FastAPI App with POST/Redirect/GET

In [ ]:
from fastapi import FastAPI, Request, Form
from fastapi.responses import RedirectResponse
from fastapi.templating import Jinja2Templates
import uvicorn, threading, json, re
from google import genai
from google.colab import userdata

# Configure Gemini
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-2.5-flash"

MOOD_PROMPT = """
Analyse the user's mood. Respond with ONLY valid JSON (no markdown):
{{"emotion":"<word>","confidence":<0-1>,"mood_description":"<2 sentences>",
"songs":[{{"title":"<t>","artist":"<a>","genre":"<g>","why":"<1 line>"}},...8 songs]}}
User text: "{mood_text}"
"""

def call_gemini(mood_text: str) -> dict:
    raw = client.models.generate_content(model=MODEL, contents=MOOD_PROMPT.format(mood_text=mood_text)).text
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        return json.loads(match.group())
    return json.loads(raw)

app3 = FastAPI()
templates3 = Jinja2Templates(directory="templates")

@app3.get("/detect")
def detect_get(request: Request, error: str = None):
    return templates3.TemplateResponse("detect.html", {"request": request, "error": error})

@app3.post("/detect")
async def detect_post(mood_text: str = Form(...)):
    if not mood_text.strip():
        return RedirectResponse("/detect?error=Please+enter+a+mood+description", status_code=303)
    try:
        result = call_gemini(mood_text.strip())
        key = cache_store(result)
        return RedirectResponse(f"/results?key={key}", status_code=303)
    except Exception as e:
        return RedirectResponse(f"/detect?error={str(e)[:80]}", status_code=303)

@app3.get("/results")
def results_get(request: Request, key: str):
    result = cache_retrieve(key)
    if not result:
        return RedirectResponse("/detect?error=Result+not+found+or+expired", status_code=303)
    return templates3.TemplateResponse("results.html", {"request": request, "result": result})

thread3 = threading.Thread(
    target=lambda: uvicorn.run(app3, host="0.0.0.0", port=8002, log_level="error"),
    daemon=True
)
thread3.start()
print("Server running on port 8002")

In [ ]:
# Test the POST flow programmatically
import requests as req

# POST to /detect — should redirect to /results?key=...
session = req.Session()
resp = session.post(
    "http://localhost:8002/detect",
    data={"mood_text": "I'm feeling excited and nervous before a big presentation!"},
    allow_redirects=True
)
print(f"Final URL: {resp.url}")
print(f"Status:    {resp.status_code}")
print("Contains emotion:", "emotion" in resp.text.lower() or "Mood:" in resp.text)

### ✏️ In-Class Exercise 5b — Add Empty Input Validation

In [ ]:
# Test empty input — it should redirect back to /detect with an error
resp_empty = session.post(
    "http://localhost:8002/detect",
    data={"mood_text": ""},
    allow_redirects=True
)
print(f"Empty input URL: {resp_empty.url}")
print("Has error:", "error" in resp_empty.url)

### ✏️ In-Class Exercise 5c — Trace the Full Flow

Complete the diagram in your own words (edit this cell).

**POST-Redirect-GET Flow:**

1. User types mood and clicks submit → browser sends **___** to **___**
2. FastAPI calls **___** with the mood text
3. Gemini returns **___** which we parse into a Python **___**
4. We store it in `_cache` with key = **___**
5. We return HTTP **___** redirect to `/results?key=...`
6. Browser follows redirect, sends **___** to `/results?key=...`
7. FastAPI reads the key, retrieves **___** from cache
8. Returns rendered **___.html** with the result data

---
## 🏠 After-Class Exercises

---
### After-Class A: Add TTL Expiry Message

In [ ]:
# TODO: When cache_retrieve returns None, check if the key exists but is expired
# Show a different error message: "Your result expired. Please detect again."
# vs "Result not found" when the key was never valid

### After-Class B: Short Mood Validation

In [ ]:
# TODO: Reject mood_text shorter than 10 characters
# Return an error: "Please describe your mood in at least 10 characters."
# Hint: add this check before calling call_gemini()

### After-Class C (Challenge): Read the Real Implementation

Open `routers/api.py` and `core/result_cache.py` from the MoodBeats AI project.
Answer these questions in the cells below:

1. How does the real `result_cache.py` handle the case where the cache gets too large?  
   _Answer:_

2. What is `_gemini(request)` in `routers/api.py` doing?  
   _Answer:_

3. The real app supports two input types (`face` and `text`). Where does `input_type` come from?  
   _Answer:_

---
## ✅ Session 5 Complete!

**You learned:**
- HTML forms and POST requests
- The POST-Redirect-GET pattern and why it matters
- In-memory UUID cache for storing results
- Wiring Gemini into a real web route

**Next session:** Music links and the results page!